# 🚀 Train an LLM from scratch (Colab)

A simple, Colab-ready project to train a small decoder-only LLM **from scratch**
(no pretrained weights) on **WikiText-103**, with selectable model size
(`tiny` / `0.1b` / `0.5b` / `1b` / `custom`) and selectable architecture
(`gpt2` style or `llama` style).


<details>
<summary>中文说明（点击展开）</summary>

一个简单、可在 Google Colab 上运行的项目：在 **WikiText-103** 上**从头**训练小型
decoder-only LLM，可选择模型尺寸 (`tiny` / `0.1b` / `0.5b` / `1b` / `custom`)
与架构 (`gpt2` 风格或 `llama` 风格)。


</details>

**Architecture options**
- `gpt2` — GPT-2 style (LayerNorm + learned absolute positions + GELU MLP)
- `llama` — LLaMA style (RMSNorm + RoPE + SwiGLU, no bias, untied embeddings)

**Size presets** (approximate parameter count with vocab=50257)

| size | n_layer | n_head | n_embd | block_size | ~params | Colab tier |
|------|---------|--------|--------|------------|---------|------------|
| tiny | 4  | 4  | 256  | 256  | ~16M  | free T4 ✅ |
| 0.1b | 8  | 12 | 768  | 512  | ~95M  | free T4 (slow) / Pro ✅ |
| 0.5b | 24 | 16 | 1280 | 1024 | ~500M | Pro / A100 ✅ |
| 1b   | 24 | 16 | 1536 | 1024 | ~750M | A100 only ⚠️ |
| custom | you set | | | | | any |

**Recommended workflow**
1. Run on free T4 with `size=tiny` to verify everything works end-to-end (~10 min).
2. Bump to `0.1b` for a coherent-text demo.
3. Use `0.5b` / `1b` only on A100 (Colab Pro / Pay-as-you-go).
4. To **continue training** from a previous run, set `RESUME_FROM = "auto"`
   in section 8 (see there for details).


## 1. Check GPU

Make sure you have a GPU runtime: **Runtime → Change runtime type → T4 GPU**
(or A100 on Pro).


<details>
<summary>中文说明（点击展开）</summary>


请确认已开启 GPU 运行时：**运行时 → 更改运行时类型 → T4 GPU**（Pro 用户可选 A100）。


</details>


In [ ]:
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")


## 2. Install dependencies

On Colab the runtime already has `torch`. We add `transformers` + `datasets`.


<details>
<summary>中文说明（点击展开）</summary>


Colab 运行时已预装 `torch`，这里追加 `transformers` 与 `datasets`。


</details>


In [ ]:
!pip install -q --upgrade transformers datasets accelerate tokenizers


## 3. Get the project files

Two options (uncomment the one you want):
- **Option A**: Clone from GitHub.
- **Option B**: Upload the `llm_scratch/` directory manually via the Colab
  file browser.


<details>
<summary>中文说明（点击展开）</summary>


两个选项（取消注释你需要的那个）：
- **A**：从 GitHub clone。
- **B**：手动上传 `llm_scratch/` 目录到 Colab 文件浏览器。


</details>


In [ ]:
# Option A: clone from GitHub
# !git clone https://github.com/BF667/llm-from-scratch.git
# %cd /content/llm-from-scratch

# Option B: assume you uploaded the llm_scratch/ folder next to this notebook.
import os, sys
if not os.path.isdir("llm_scratch"):
    if os.path.isdir("/content/llm_scratch"):
        %cd /content
print("cwd:", os.getcwd())
print("llm_scratch exists:", os.path.isdir("llm_scratch"))
sys.path.insert(0, os.getcwd())


## 4. Select architecture & size 👈

**This is the main "select parameters" cell.** Edit the two variables below.

| `ARCH` | description |
|--------|-------------|
| `gpt2` | GPT-2 style (LayerNorm + GELU + learned positions) |
| `llama`| LLaMA style (RMSNorm + RoPE + SwiGLU) |

| `SIZE` | description |
|--------|-------------|
| `tiny` | ~16M params, Colab free T4, ~10 min |
| `0.1b` | ~95M, small but coherent |
| `0.5b` | ~500M, needs A100 / Pro |
| `1b`   | ~750M, A100 only |
| `custom` | you set `n_layer / n_head / n_embd / block_size` |


<details>
<summary>中文说明（点击展开）</summary>


**这是主要的"选择参数"单元格。** 修改下面两个变量即可。


</details>


In [ ]:
# ============== EDIT HERE ==============
ARCH = "gpt2"          # "gpt2" or "llama"
SIZE = "tiny"          # "tiny" | "0.1b" | "0.5b" | "1b" | "custom"
# ======================================

# If SIZE == "custom", set these:
CUSTOM = {
    "n_layer": 6,
    "n_head": 6,
    "n_embd": 384,
    "block_size": 512,
}

# Training overrides (optional — None means use size preset default)
TRAIN = {
    "epochs": 1,         # number of epochs
    "batch_size": None,  # None = use preset
    "grad_accum": None,
    "lr": None,
    "subset_size": None, # None = full dataset; set small int for smoke test
    "grad_checkpoint": False,
}

assert ARCH in ("gpt2", "llama"), f"bad ARCH={ARCH!r}"
assert SIZE in ("tiny", "0.1b", "0.5b", "1b", "custom"), f"bad SIZE={SIZE!r}"
print(f"Selected: arch={ARCH}, size={SIZE}")
if SIZE == "custom":
    print(f"Custom config: {CUSTOM}")


## 5. Preview config & parameter count


In [ ]:
from llm_scratch.config import build_config, estimate_params, list_presets

overrides = CUSTOM if SIZE == "custom" else {}
cfg = build_config(arch=ARCH, size=SIZE, **overrides)

print("Config:")
for k, v in cfg.items():
    if k != "training":
        print(f"  {k}: {v}")
print("Training:")
for k, v in cfg["training"].items():
    print(f"  {k}: {v}")
print(f"\nEstimated params: ~{estimate_params(cfg)/1e6:.1f}M")
print(f"VRAM needed (rough): ~{estimate_params(cfg)*4*4/1e9:.2f} GB for weights+optimizer (fp32 Adam)")


## 6. Load tokenizer + WikiText-103

We use the GPT-2 BPE tokenizer (vocab=50257) so we don't need to train one.
First run downloads ~180MB and caches it; subsequent runs are instant.


<details>
<summary>中文说明（点击展开）</summary>


使用 GPT-2 BPE tokenizer（词表 50257），无需自训 tokenizer。
首次运行会下载约 180MB 并缓存，后续运行秒级完成。


</details>


In [ ]:
from llm_scratch.dataset import build_tokenizer, load_wikitext_dataset

tokenizer = build_tokenizer("gpt2")
print(f"Tokenizer: {tokenizer.__class__.__name__}, vocab={len(tokenizer)}")

# subset_size=None = full WikiText-103. For a quick demo, set e.g. 2000 to subsample.
subset = TRAIN.get("subset_size")
print(f"Loading WikiText-103 (subset={subset}) ...")
ds = load_wikitext_dataset(tokenizer, block_size=cfg["block_size"], subset_size=subset)
print(f"train rows: {len(ds['train'])}, val rows: {len(ds['validation'])}, test rows: {len(ds['test'])}")
print(f"example input_ids[0][:16]: {ds['train'][0]['input_ids'][:16]}")
print(f"example text: {tokenizer.decode(ds['train'][0]['input_ids'][:60])!r}")


## 7. Build the model


In [ ]:
from llm_scratch.train import build_model_from_config

model, mcfg = build_model_from_config(cfg)
n = sum(p.numel() for p in model.parameters())
print(f"Built {ARCH} / {SIZE} model")
print(f"Actual params: {n/1e6:.2f}M")
print(f"Config: {mcfg}")

if TRAIN.get("grad_checkpoint", False):
    model.gradient_checkpointing_enable()
    print("Gradient checkpointing ENABLED.")


## 8. Continue training from a previous run 🔄

Set `RESUME_FROM` to continue training from a checkpoint. HF `Trainer` will
restore the **model weights, optimizer state, LR scheduler, RNG, and global
step counter** — so the model keeps learning from where it stopped, instead
of restarting from random init.

| `RESUME_FROM` value | meaning |
|---------------------|---------|
| `None`              | Train from scratch (random init) — **default** |
| `"auto"` or `True`  | Auto-pick the latest `checkpoint-<step>` subdir inside `output_dir` |
| `"/path/to/checkpoint-XYZ"` | Resume from that specific checkpoint directory |

**Typical workflow for continue-training:**
1. Run section 9 once with `RESUME_FROM = None` to do the first training pass.
2. Bump `TRAIN["epochs"]` to a higher number (or keep 1 — Trainer adds to the
   existing step count), set `RESUME_FROM = "auto"`, and re-run section 9.
3. The loss should pick up where it left off, not jump back to ~10.

> ⚠️ The architecture & size must match the checkpoint — you can't resume a
> `gpt2-tiny` checkpoint with `ARCH="llama"` or `SIZE="0.1b"`.


<details>
<summary>中文说明（点击展开）</summary>


设置 `RESUME_FROM` 即可从 checkpoint 续训。HF `Trainer` 会恢复**模型权重、
优化器状态、学习率调度、随机数状态与全局步数**，模型从上次停下的地方继续学习，
而不是从随机初始化重启。

| `RESUME_FROM` 值 | 含义 |
|------------------|------|
| `None`           | 从头训练（随机初始化）—— **默认** |
| `"auto"` 或 `True` | 自动选 `output_dir` 中最新的 `checkpoint-<step>` 子目录 |
| `"/path/to/checkpoint-XYZ"` | 从指定 checkpoint 目录续训 |

**续训典型流程：**
1. 先用 `RESUME_FROM = None` 跑一次第 9 节，完成首次训练。
2. 把 `TRAIN["epochs"]` 调高（或保持 1，Trainer 会在已有步数上累加），
   设 `RESUME_FROM = "auto"`，重新跑第 9 节。
3. loss 应当从上次的位置继续下降，而不是跳回 ~10。

> ⚠️ 架构与尺寸必须与 checkpoint 一致 —— 不能用 `ARCH="llama"` 或
> `SIZE="0.1b"` 去续训一个 `gpt2-tiny` 的 checkpoint。


</details>


In [ ]:
# ============== CONTINUE-TRAINING OPTION ==============
# None               -> train from scratch
# "auto" or True     -> auto-resume from latest checkpoint in output_dir
# "<path>"           -> resume from that specific checkpoint directory
RESUME_FROM = None
# ======================================================

print(f"RESUME_FROM = {RESUME_FROM!r}")
if RESUME_FROM is None:
    print("Mode: train from scratch / 从头训练")
elif RESUME_FROM in ("auto", True):
    print("Mode: auto-resume from latest checkpoint / 自动续训")
else:
    print(f"Mode: resume from {RESUME_FROM}")


## 9. Train 🏋️

Uses HF `Trainer` under the hood. The first 200 steps of a from-scratch run
usually show loss dropping from ~10 (uniform over vocab) toward 4-5 on tiny
models, lower on bigger ones. If you set `RESUME_FROM` above, the loss will
pick up where the previous run stopped.


<details>
<summary>中文说明（点击展开）</summary>


底层使用 HF `Trainer`。从头训练的前 200 步 loss 通常会从 ~10（词表均匀分布）
降到 4-5（tiny 模型），大模型更低。如果上面设置了 `RESUME_FROM`，loss 会从
上次停下的位置继续。


</details>


In [ ]:
from llm_scratch.train import run_training
import os

output_dir = f"out/{ARCH}-{SIZE}"
print(f"Output dir: {output_dir}")

# Show whether there are existing checkpoints to resume from (informational).
if os.path.isdir(output_dir):
    ckpts = sorted([d for d in os.listdir(output_dir) if d.startswith("checkpoint-")])
    print(f"Existing checkpoints in {output_dir}: {ckpts if ckpts else '(none)'}")
else:
    print(f"No output dir yet — first run.")

final_dir = run_training(
    arch=ARCH,
    size=SIZE,
    output_dir=output_dir,
    epochs=TRAIN.get("epochs"),
    batch_size=TRAIN.get("batch_size"),
    grad_accum=TRAIN.get("grad_accum"),
    lr=TRAIN.get("lr"),
    subset_size=TRAIN.get("subset_size"),
    grad_checkpoint=TRAIN.get("grad_checkpoint", False),
    resume_from_checkpoint=RESUME_FROM,
    **(CUSTOM if SIZE == "custom" else {}),
)
print(f"\nTraining done. Model saved to: {final_dir}")


## 10. Generate text ✍️


In [ ]:
import torch
from llm_scratch.generate import load_model
from transformers import AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(final_dir)
model = load_model(final_dir).to(device)
model.eval()

PROMPTS = [
    "The future of artificial intelligence is",
    "In a galaxy far, far away,",
    "Once upon a time,",
    "Machine learning is a field of",
]

for prompt in PROMPTS:
    ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    out = model.generate(
        ids,
        max_new_tokens=80,
        temperature=0.8,
        top_k=50,
        do_sample=True,
    )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"\n--- PROMPT: {prompt!r} ---")
    print(text)


## 11. (Optional) Save to Google Drive

Mount your Drive and copy the trained model there so it survives runtime
disconnects.


<details>
<summary>中文说明（点击展开）</summary>


挂载 Drive 并把训练好的模型复制过去，避免运行时断开后丢失。


</details>


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r {final_dir} /content/drive/MyDrive/llm-from-scratch/
# print("Saved to Drive.")


## 12. CLI alternative

You can also run training from the terminal (Colab: prepend `!`):

```bash
# Tiny GPT-2 on a 2000-row subset for a quick smoke test
python -m llm_scratch.train --arch gpt2 --size tiny --output_dir out/gpt2-tiny \
    --subset_size 2000 --epochs 1

# LLaMA 0.1B on the full WikiText-103
python -m llm_scratch.train --arch llama --size 0.1b --output_dir out/llama-0.1b

# Custom size — override individual knobs
python -m llm_scratch.train --arch gpt2 --size custom \
    --n_layer 12 --n_head 12 --n_embd 768 --block_size 768 \
    --output_dir out/gpt2-custom --grad_checkpoint

# Continue training from the latest checkpoint in out/gpt2-tiny
python -m llm_scratch.train --arch gpt2 --size tiny \
    --output_dir out/gpt2-tiny --epochs 2 \
    --resume_from_checkpoint auto

# Continue training from a specific checkpoint directory
python -m llm_scratch.train --arch gpt2 --size tiny \
    --output_dir out/gpt2-tiny --epochs 2 \
    --resume_from_checkpoint out/gpt2-tiny/checkpoint-500

# Generate
python -m llm_scratch.generate --checkpoint out/gpt2-tiny \
    --prompt "The future of AI is" --max_new_tokens 80
```


<details>
<summary>中文说明（点击展开）</summary>


也可以从命令行运行训练（Colab 中加 `!` 前缀）。新增 `--resume_from_checkpoint`
参数：值为 `auto` 自动找最新 checkpoint，或传入 checkpoint 目录路径。


</details>
